In [18]:
import sklearn
import numpy as np
import pandas as pd
import os
import re 
import sys 
import importlib
import datetime as dt

############ LOAD in custom packages ################

project_root = os.path.join(os.getcwd(), "..") # Get path of the project 
sys.path.append(project_root) # Add project root to sys.path for script usage

# Import and reload (optional) custom scripts
from scripts import paths
from scripts import preprocessing as pre
from scripts import visualization as vis
from scripts import variables
from scripts import feature_selection as fs

importlib.reload(paths)
importlib.reload(pre)
importlib.reload(vis)
importlib.reload(variables)
importlib.reload(fs)


# Define label variables
df_names = ['v1_day', 'v2_day', 'v1_week', 'v2_week']



# Filepaths
brighten_dir = paths.DATA
sub_dir = paths.SUB_DFS
results_dir = paths.RESULTS
demo_dir = paths.DEMO

In [38]:
# Create binary of depression score at the end of the time
count=0
start_depressed=np.nan
end_depressed=np.nan
last_phq9=np.nan
change_bin=np.nan
for name in ['v1_day','v2_day']: 
	phq9_end = []
	Xy = pd.read_csv(os.path.join(brighten_dir, f'{name}_trainval.csv'))
	for sub, sub_df in Xy.groupby('num_id'):
		sub_phq9 = sub_df.dropna(subset='phq9_sum')
		if len(sub_phq9) == 0:
			continue
		sub_phq9 = sub_phq9.sort_values(by='day', ascending=True)

		first_phq9 = list(sub_phq9['phq9_sum'])[0]
		if first_phq9 > 10:
			start_depressed = 1
		else:
			start_depressed = 0


		# Phq9 at ~6 weeks
		days6weeks=sub_phq9[sub_phq9['day']>38]
		days6weeks=days6weeks[days6weeks['day']<55]
		if not len(days6weeks) > 0:
			phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin]) #np.nan for last_phq9, end_depressed
			continue
		days6weeks_cols = days6weeks.dropna(how='all', axis=1)
		if not 'phq9_sum' in days6weeks_cols:
			phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin]) #np.nan for last_phq9, end_depressed
			continue

		last_phq9 = list(days6weeks['phq9_sum'])[0]
		if last_phq9 > 10:
			end_depressed = 1
		else:
			end_depressed = 0
		
		change_bin = start_depressed - end_depressed

		# Make 'Total Group' Variable:
			# 0: Start not-depressed, end not-depressed
			# 1: Start not-depressed, end depressed
			# 2: Start depressed, end not-depressed
			# 3: Start depressed, end depressed
		if start_depressed+end_depressed == 0:
			dep_group = 0
			stayed_depressed = 1
		if start_depressed == 0 and end_depressed == 1:
			dep_group = 1
		if start_depressed == 1 and end_depressed == 0:
			dep_group = 2
		if start_depressed+end_depressed == 2:
			dep_group = 3
			stayed_depressed = 2


		# if count < 3:
		# 	count+=1
		# 	print(f'Sub: {sub}')
		# 	display(days6weeks[['day','dt','phq9_sum']])
		# 	display(f'day: {list(days6weeks['day'])[0]}, phq9: {list(days6weeks['phq9_sum'])[0]}')
		

		phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin, dep_group, stayed_depressed])

	phq9_end_df = pd.DataFrame(phq9_end, columns=['num_id', 'phq9_sum_start', 'start_depressed_binary', 'phq9_sum_6wks', '6wks_depressed_binary', 'depression_change_bin', 'dep_group','stayed_depressed'])

	phq9_end_df.to_csv(os.path.join(demo_dir, f'{name}_phq9sum_6wks.csv'), index=False)
	#display(phq9_end_df)
	print(f'\n\nSaved phq9_end_df to {name}_phq9sum_6wks.csv')
	print(phq9_end_df['start_depressed_binary'].value_counts())
	print(phq9_end_df['6wks_depressed_binary'].value_counts())
	print(phq9_end_df['depression_change_bin'].value_counts())
	print(phq9_end_df['dep_group'].value_counts())



print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))



Saved phq9_end_df to v1_day_phq9sum_6wks.csv
start_depressed_binary
0    106
1     73
Name: count, dtype: int64
6wks_depressed_binary
0    109
1     70
Name: count, dtype: int64
depression_change_bin
 0    133
-1     23
 1     23
Name: count, dtype: int64
dep_group
0.0    72
3.0    39
2.0    21
1.0    19
Name: count, dtype: int64


Saved phq9_end_df to v2_day_phq9sum_6wks.csv
start_depressed_binary
1    84
0    62
Name: count, dtype: int64
6wks_depressed_binary
1    75
0    71
Name: count, dtype: int64
depression_change_bin
 0    105
 1     26
-1     15
Name: count, dtype: int64
dep_group
3.0    28
0.0    26
2.0    14
1.0     5
Name: count, dtype: int64
Run on: Thu 28 May 2026, 05:14PM


In [39]:
v1_phq9_end_df=pd.read_csv(os.path.join(demo_dir, f'v1_day_phq9sum_6wks.csv'))
v2_phq9_end_df=pd.read_csv(os.path.join(demo_dir, f'v2_day_phq9sum_6wks.csv'))
phq9_end_df = pd.concat([v1_phq9_end_df, v2_phq9_end_df], axis=0)
phq9_end_df = phq9_end_df.loc[:, ~phq9_end_df.columns.str.contains('Unnamed')]
display(phq9_end_df)

phq9_end_df.to_csv(os.path.join(demo_dir, f'phq9sum_6wks.csv'), index=False)

,num_id,phq9_sum_start,start_depressed_binary,phq9_sum_6wks,6wks_depressed_binary,depression_change_bin,dep_group,stayed_depressed
0,13.0,9.0,0,15.0,1,-1,1.0,1.0
1,14.0,10.0,0,12.0,1,-1,1.0,1.0
2,17.0,18.0,1,12.0,1,-1,NaN,NaN
3,29.0,12.0,1,7.0,0,1,2.0,1.0
4,44.0,9.0,0,7.0,0,0,0.0,1.0
...,...,...,...,...,...,...,...,...
141,1081.0,4.0,0,7.0,0,0,0.0,1.0
142,1096.0,19.0,1,20.0,1,0,3.0,2.0
143,1100.0,9.0,0,20.0,1,0,NaN,NaN
144,1101.0,4.0,0,20.0,1,0,NaN,NaN


In [44]:
print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))

import plotly.express as px

for name in ['v1_week', 'v2_week']:
	print(f'################# {name} ##################')
	train = pd.read_csv(os.path.join(brighten_dir, f'{name}_trainval.csv'))
	change_in_phq9_cat = []
	for i, sub_df in train.groupby('num_id'):
			sub_df=sub_df.dropna(subset='phq9_sum')
			sub_df = sub_df.sort_values('day')
			sub_df['phq9_sum'] = pd.to_numeric(sub_df['phq9_sum'])
			sub_df['phq9_cat'] = pd.cut(sub_df['phq9_sum'],
							  bins=[0, 5, 10, 15, 20, 27],
							  labels=[0,1,2,3,4], #low, med-low, med, med-high, high
							  right=False,
							  include_lowest=True)

			if sub_df.shape[0]>0:
				og_cat = sub_df.iloc[0]['phq9_cat']
				end_cat = sub_df.iloc[sub_df.shape[0]-1]['phq9_cat']
				try: 
					cat_diff = end_cat-og_cat
					weeks = sub_df['week'].max()
					if weeks > 4:
						if cat_diff < 0:
							 diff_cat = -1
						elif cat_diff > 0:
							 diff_cat = 1
						else: 
							 diff_cat = 0
						change_in_phq9_cat.append([i, cat_diff, weeks, diff_cat])
						 
				except Exception as e:
					print(i, e)
					continue
	
	# change 

	change_df = pd.DataFrame(change_in_phq9_cat, columns=['sub', 'change', 'weeks', 'change_category'])
	change_df['name'] = name
	change_df.to_csv(os.path.join(demo_dir, f'{name}_change_in_phq9_cat.csv'))
	#display(change_df)


	fig = px.histogram(change_df, x='change', color='weeks', barmode='stack', title=f'{name.upper()} Raw PHQ9_Sum-Category-Change: Base to Final Week')
	fig.show()

	fig2 = px.histogram(change_df, x='change_category', color='weeks', barmode='stack', title=f'{name.upper()} Change_category of PHQ9_Sum-Category-Change: Base to Final Week')
	fig2.show()



Run on: Thu 28 May 2026, 05:16PM
################# v1_week ##################


################# v2_week ##################


In [45]:
# Merge in 
for name in df_names:
	df = pd.read_csv(os.path.join(brighten_dir, f'{name}_trainval_pca.csv'), low_memory=False)

	if 'v1' in name:
		phq9_change = pd.read_csv(os.path.join(demo_dir, 'v1_day_phq9sum_6wks.csv')).rename(columns={'week':'ending_week'})
	else:
		phq9_change = pd.read_csv(os.path.join(demo_dir, 'v2_day_phq9sum_6wks.csv')).rename(columns={'week':'ending_week'})


	phq9_change=phq9_change.drop(columns=[col for col in phq9_change.columns if 'Unnamed' in col])
	merge_df = df.merge(phq9_change, on=['num_id'],how='outer')
	
	if 'v1' in name:
		change_df=pd.read_csv(os.path.join(demo_dir, f'v1_week_change_in_phq9_cat.csv'))
	else:
		change_df=pd.read_csv(os.path.join(demo_dir, f'v2_week_change_in_phq9_cat.csv'))

	change_df = change_df.rename(columns={'sub':'num_id', 'change':'phq9_cat_change', 'weeks':'total_weeks_reported','change_category':'phq9_cat_change_cat'})
	merge_df = merge_df.merge(change_df, on=['num_id'],how='outer')

	merge_df.to_csv(os.path.join(brighten_dir, f'{name}_trainval_pca_outcomes.csv'), index=False)

print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))


Run on: Thu 28 May 2026, 05:17PM


In [46]:
for name in df_names:
	print(f'\n\n{name.upper()}')
	merge_df=pd.read_csv(os.path.join(brighten_dir, f'{name}_trainval_pca_outcomes.csv'))
	for cat, cat_df in merge_df.groupby('phq9_cat_change_cat'):
		rows = len(cat_df)
		subs = cat_df['num_id'].nunique()
		print(f'For mood change {cat}: {subs} subs, {rows} observations')
	
	for gp, gp_df in merge_df.groupby('stayed_depressed'):
		rows = len(gp_df)
		subs = gp_df['num_id'].nunique()
		print(f'Among people who stayed in their mood groups, those who stayed depressed (2) vs those who stayed healthy/nondepressed (1): \n {gp}: {subs} subs, {rows} observations')




V1_DAY
For mood change -1.0: 83 subs, 5014 observations
For mood change 0.0: 59 subs, 4119 observations
For mood change 1.0: 15 subs, 1080 observations
Among people who stayed in their mood groups, those who stayed depressed (2) vs those who stayed healthy/nondepressed (1): 
 1.0: 101 subs, 7904 observations
Among people who stayed in their mood groups, those who stayed depressed (2) vs those who stayed healthy/nondepressed (1): 
 2.0: 50 subs, 3645 observations


V2_DAY
For mood change -1.0: 46 subs, 2911 observations
For mood change 0.0: 39 subs, 2145 observations
For mood change 1.0: 19 subs, 1070 observations
Among people who stayed in their mood groups, those who stayed depressed (2) vs those who stayed healthy/nondepressed (1): 
 1.0: 33 subs, 2619 observations
Among people who stayed in their mood groups, those who stayed depressed (2) vs those who stayed healthy/nondepressed (1): 
 2.0: 40 subs, 2868 observations


V1_WEEK
For mood change -1.0: 83 subs, 972 observations
For m